# TMT data treatment

This code will work with the TMT data provided by the FGCZ.

Initial working file for each experiment (hTERT_HME1_1, hTERT_HME1_2, HEK293T_1) is the raw_abundances_matrix

**First step is to rename the files to have a unified labeling system** -> (CellLine)_ (Data:Type)_ (Treatment)_ (TimePoint)_ (Replicate)

All the data transformation and statistics I am going to do starting from that initial "raw_data" file.



In [2]:
import pandas as pd

from src.column_spec import *
from src.TMT_transformations import *

hme1_1 = pd.read_csv("../../data/hme1_1_raw_sample.tsv", sep="\t")
hme1_2 = pd.read_csv("../../data/hme1_2_raw_sample.tsv", sep="\t")
hek_1 = pd.read_csv("../../data/hek_1_raw_sample.tsv", sep="\t")

## Data transformations

Run the full transformation pipeline on each dataset:
- n:reps
- raw:mean / raw:median / raw:sd / raw:cv
- log2:abs (zeros treated as NaN)
- log2:mean / log2:median / log2:sd
- log2:FC (fold change vs. starve)
- log2:scaled (max-normalised fold change, amplitude between -1 and 1)
- log2:pvalue (Welch t-test vs. starve)
- log2:FDR (Benjamini-Hochberg correction per treatment × timepoint)
- log2:adjustedFDR (-log10 transformation of the FDR, also called adjusted FDR or adjusted p-value)


In [3]:
hme1_1_transformed = run_all_transformations(hme1_1, cell_line="WT", data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hme1_2_transformed = run_all_transformations(hme1_2, cell_line="WT", data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hek_1_transformed  = run_all_transformations(hek_1,  cell_line="WT", data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])

print(f"hme1_1: {hme1_1.shape} -> {hme1_1_transformed.shape}")
print(f"hme1_2: {hme1_2.shape} -> {hme1_2_transformed.shape}")
print(f"hek_1:  {hek_1.shape}  -> {hek_1_transformed.shape}")

[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Columns: 90 -> 418
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Columns: 106 -> 434
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Columns: 106 -> 434
hme1_1: (100, 90) -> (100, 418)
hme1_2: (100, 106) -> (100, 434)
hek_1:  (100, 106)  -> (100, 434)


In [4]:
hme1_1_transformed

,protein_Id,protein_name,CON,REV,site,isotopeLabel,WT_raw:abs_EGF_full_r1,WT_raw:abs_EGF_full_r2,WT_raw:abs_EGF_full_r3,WT_raw:abs_EGF_full_r4,...,WT_log2:adjustedFDR_INS_2,WT_log2:adjustedFDR_INS_5,WT_log2:adjustedFDR_INS_10,WT_log2:adjustedFDR_INS_90,WT_log2:adjustedFDR_EGFnINS_full,WT_log2:adjustedFDR_EGFnINS_1,WT_log2:adjustedFDR_EGFnINS_2,WT_log2:adjustedFDR_EGFnINS_5,WT_log2:adjustedFDR_EGFnINS_10,WT_log2:adjustedFDR_EGFnINS_90
0,Q96S55,WRNIP1,False,False,Q96S55_465_472_1_0~SGQSYSPSR,light,118024.317900,NaN,93797.261380,103691.150600,...,0.516699,0.596945,1.119176,0.938881,0.632597,0.955339,0.797705,1.144218,2.260461,2.106752
1,Q9UHX1,PUF60,False,False,Q9UHX1_555_558_1_1_S558~FDNSDLsA,light,5371.386934,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q7L4E1,MIGA2,False,False,Q7L4E1_218_228_1_0~NPETASEPLSEPESQR,light,NaN,4577.117648,4365.393912,3355.178514,...,0.062689,0.361269,0.196960,0.047871,0.052905,0.149951,0.138513,0.316827,0.231021,0.210540
3,Q5VUA4,ZNF318,False,False,Q5VUA4_527_527_1_1_S527~sFPDIEDEEK,light,NaN,89798.344230,69621.607100,NaN,...,0.298871,0.060853,0.276143,0.333128,0.520999,0.735799,0.117751,0.080379,0.429514,0.591372
4,Q9BZF1,OSBPL8,False,False,Q9BZF1_63_71_1_0~DLHQPSLSPASPHSQGFER,light,NaN,NaN,22089.853850,22570.776450,...,0.062689,0.194838,0.172634,0.056691,0.304086,0.281405,0.377123,0.174766,0.122110,0.348188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,O15400,STX7,False,False,O15400_75_81_1_1_T79~EFGSLPTtPSEQR,light,80688.790360,82783.928980,NaN,NaN,...,0.298871,0.877032,0.050820,0.442475,0.664409,0.849832,0.123231,1.833833,0.148972,0.591372
96,P54760,EPHB4,False,False,P54760_970_987_1_0~SQAKPGTPGGTGGPAPQY,light,308989.353400,336899.447900,320064.222600,325597.636500,...,0.336398,0.036789,0.507008,0.688723,0.086510,1.271633,0.425024,0.770617,0.382861,1.188519
97,P20042,EIF2S2,False,False,P20042_2_13_1_1_S2~sGDEMIFDPTMSK,light,140547.331200,179411.331700,129126.824700,144536.113000,...,0.316297,0.779160,0.490799,0.968771,0.294679,1.769768,0.945549,1.715516,0.629976,1.490275
98,Q9Y5B0,CTDP1,False,False,Q9Y5B0_469_478_1_0~SSSSASDGESEGKR;SSSSASDGESEG...,light,NaN,90995.270640,81217.088620,91233.496810,...,0.443337,0.290460,0.151623,0.607345,0.194346,0.542859,0.377123,0.938628,0.111216,0.427002


In [5]:
# Save transformed datasets
hek_1_transformed.to_csv("../../data/hek_1_transformed.tsv", sep="\t", index=False)
hme1_1_transformed.to_csv("../../data/hme1_1_transformed.tsv", sep="\t", index=False)
hme1_2_transformed.to_csv("../../data/hme1_2_transformed.tsv", sep="\t", index=False)